# 01. 元データ収集

OFAC SDN / BIS Entity List / e-Gov 貨物等省令 を公式ソースから取得して `data/staging/` に保存する。

| # | データソース | 取得先 | 出力先 |
|---|------------|--------|--------|
| 1 | OFAC SDN XML | treasury.gov | staging/sanctions/ofac_sdn_raw.json |
| 2 | BIS Entity List | trade.gov CSL API | staging/sanctions/bis_el_raw.json |
| 3 | e-Gov 貨物等省令 | elaws.e-gov.go.jp | staging/fefta/fefta_articles_raw.json |

In [ ]:
# ── 設定 ─────────────────────────────────────────────────────────────────────
# DRY_RUN=True の場合、ネットワーク取得は行うが data/staging/ への書き込みはスキップ
DRY_RUN = False  # 本番実行時は False に変更

import sys, os
from pathlib import Path

# パス（00_setup.ipynb を先に実行した場合は変数が引き継がれる）
try:
    BASE
except NameError:
    BASE        = Path("/Users/takehirosato/Desktop/AI_TradeManagement")
    STAGING_DIR = BASE / "data" / "staging"
    sys.path.insert(0, str(BASE / "scripts"))

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
print(f"DRY_RUN={DRY_RUN}, STAGING_DIR={STAGING_DIR}")

## 1. OFAC SDN XML 取得

In [ ]:
# OFAC SDN 取得 (~34MB, 認証不要)
from pipeline.collect.ofac_sdn import fetch_ofac_sdn

ofac_cache = STAGING_DIR / "sanctions" / "ofac_sdn_raw.json"
ofac_entities = fetch_ofac_sdn(
    cache_path=None if DRY_RUN else ofac_cache
)
print(f"✅ OFAC SDN: {len(ofac_entities):,} エンティティ取得")
print(f"   例: {ofac_entities[0] if ofac_entities else '(none)'}")

## 2. BIS Entity List 取得

In [ ]:
# BIS Entity List (Trade.gov CSL API)
from pipeline.collect.bis_entity_list import fetch_bis_entity_list

try:
    BIS_API_KEY
except NameError:
    BIS_API_KEY = os.environ.get("BIS_API_KEY", "DEMO_KEY")

bis_cache = STAGING_DIR / "sanctions" / "bis_el_raw.json"
bis_entities = fetch_bis_entity_list(
    api_key=BIS_API_KEY,
    cache_path=None if DRY_RUN else bis_cache,
)
print(f"✅ BIS Entity List: {len(bis_entities):,} エンティティ取得")
print(f"   例: {bis_entities[0] if bis_entities else '(none)'}")

## 3. e-Gov 貨物等省令 取得

In [ ]:
# e-Gov 貨物等省令 XML
from pipeline.collect.egov_fefta import fetch_fefta_ministerial_ordinance

fefta_cache = STAGING_DIR / "fefta" / "fefta_articles_raw.json"
fefta_articles = fetch_fefta_ministerial_ordinance(
    cache_path=None if DRY_RUN else fefta_cache,
)
if fefta_articles:
    print(f"✅ 貨物等省令: {len(fefta_articles)} 条文取得")
    print(f"   例: {fefta_articles[0]}")
else:
    print("⚠️  条文取得なし — e-Gov API の応答を確認してください。")
    print("    手動取得の場合: data/staging/fefta/fefta_articles_raw.json に配置してください。")

## 4. 取得結果サマリー

In [ ]:
# サマリー
print("=" * 50)
print("取得結果サマリー")
print("=" * 50)
print(f"OFAC SDN       : {len(ofac_entities):>6,} 件")
print(f"BIS Entity List: {len(bis_entities):>6,} 件")
print(f"貨物等省令条文  : {len(fefta_articles):>6,} 件")
print()
print("staging 出力ファイル:")
for f in sorted((STAGING_DIR).rglob("*.json")):
    size = f.stat().st_size
    print(f"  {f.relative_to(STAGING_DIR)}  ({size:,} bytes)")
print()
print("次のノートブック → 02_integrate_datasets.ipynb")